In [ ]:
# Cell 1: Environment Setup & Path Linking

!pip install shap -q

from google.colab import drive
drive.mount('/content/drive')

import sys, os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# THE GHOST PURGE: Explicitly force Eager Execution OFF to fix the Keras Graph
tf.config.run_functions_eagerly(False)

# Define Paths
PROJECT_ROOT = '/content/drive/MyDrive/AIKONIC-AI'
PHASE_2_DIR  = os.path.join(PROJECT_ROOT, 'PHASE_02')
PHASE_3_DIR  = os.path.join(PROJECT_ROOT, 'PHASE_03')
MODEL_PATH   = os.path.join(PROJECT_ROOT, 'checkpoints', 'FINAL_production_model.keras')

os.makedirs(PHASE_3_DIR, exist_ok=True)
%cd {PHASE_3_DIR}

# Link Phase 2 and Phase 3 modules
if PHASE_2_DIR not in sys.path: sys.path.insert(0, PHASE_2_DIR)
if PHASE_3_DIR not in sys.path: sys.path.insert(0, PHASE_3_DIR)

from data_loader import DataLoader
import config
import explainability as xai

# Load Model
print("\nLoading Final Production Model...")
prod_model = tf.keras.models.load_model(MODEL_PATH)
print("✓ Model Loaded Successfully!")

# Initialize Data Loader
print("Initializing DataLoader...")
loader = DataLoader()
X_test, y_test, test_paths = loader.get_arrays("test", return_paths=True)
print("✓ Environment Ready. Proceed to Cell 2.")

# Force Colab to reload the module so it catches our new manual override!
import importlib
importlib.reload(xai)
print("✓ XAI Module strictly reloaded!")

In [ ]:
# Cell 2: Generate the 4-Panel Visualization & Clinical Narrative

# 1. Select a patient patch
IMAGE_INDEX = 159

test_image = X_test[IMAGE_INDEX]
true_label = y_test[IMAGE_INDEX]

# 2. Get AI Prediction
test_image_expanded = np.expand_dims(test_image, axis=0)
prediction = prod_model.predict(test_image_expanded, verbose=0)
predicted_class = config.CLASS_NAMES[np.argmax(prediction)]
confidence = np.max(prediction) * 100

# Convert the integer true_label back to a readable string
true_class_name = config.CLASS_NAMES[true_label]

# 3. Generate XAI Math
heatmap = xai.make_gradcam_heatmap(test_image_expanded, prod_model)
shap_values = xai.generate_shap_values(prod_model, test_image_expanded)

# 4. Generate & Print Automated Clinical Narrative
clinical_text = xai.generate_clinical_narrative(
    img_array=test_image,
    shap_values=shap_values,
    predicted_class=predicted_class,
    confidence=confidence
)

print("="*80)
print(f"GROUND TRUTH (ACTUAL DIAGNOSIS): {true_class_name}")
print("="*80)
print(clinical_text)
print("="*80)

# 5. Render the 4-Panel Graphic
xai.plot_4_panel_diagnostic(
    img_array=test_image,
    heatmap=heatmap,
    shap_values=shap_values,
    save_name=f"diagnostic_patch_{IMAGE_INDEX}.png"
)